# Feature engineering

**Lê:** `outputs/data/clean_hourly.csv`

**Grava:** `outputs/data/modeling_frame.csv` + `split_info.json`

Split cronológico 80/20, `random_state=67`, mesmas features RF+PLS.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

# sobe até achar analyses/grupo2
here = Path.cwd().resolve()
group = None
for c in [here, *here.parents]:
    if (c / 'config.py').exists() and (c / 'lib').is_dir():
        group = c
        break
if group is None:
    raise RuntimeError('Abra/rode este notebook de dentro de analyses/grupo2/...')
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib
from lib import features, eda_stl, models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print('grupo2 →', group)
print('outputs →', config.OUTPUT_DIR)


In [ ]:
clean_path = require_file(config.DATA_DIR / 'clean_hourly.csv', '01_dados/02_limpeza_preparacao.ipynb')
clean = pd.read_csv(clean_path, parse_dates=[config.DATETIME_COL])
feat = features.add_calendar_and_lags(clean)
model_df = features.modeling_matrix(feat)
train, test, cut = data_lib.chronological_split(model_df)
data_lib.save_frame(model_df, config.DATA_DIR / 'modeling_frame.csv')
info = {
    'n_modeling': len(model_df),
    'train_size': len(train),
    'test_size': len(test),
    'train_ratio': config.TRAIN_RATIO,
    'cut_index': cut,
    'train_start': str(train[config.DATETIME_COL].iloc[0]),
    'train_end': str(train[config.DATETIME_COL].iloc[-1]),
    'test_start': str(test[config.DATETIME_COL].iloc[0]),
    'test_end': str(test[config.DATETIME_COL].iloc[-1]),
    'random_state': config.RANDOM_STATE,
    'horizon': config.HORIZON,
    'features': config.FEATURE_COLS,
}
data_lib.save_json(info, config.DATA_DIR / 'split_info.json')
print('features:', config.FEATURE_COLS)
display(info)
display(model_df.head())